In [ ]:
import pandas as pd
from dataset_class.job_post_dataset import JobPostingDataset
from sklearn.model_selection import train_test_split
import nltk
from nltk.tokenize import word_tokenize
from collections import Counter
import numpy as np
from gensim.models import FastText, Word2Vec
from itertools import product
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'
os.environ['PYTHONWARNINGS'] = 'ignore'


nltk.download('punkt')
nltk.download('punkt_tab')


### Splitting 

This section serves to split the dataset into Train, validation and test sets

In [ ]:
### loading clean text
df = pd.read_csv("./data/clean/fake_job_postings_nlp.csv")
df.head()


In [ ]:
# 1. Split 80% Train, 20% "Rest" (temp_data)
train_data, temp_data = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df['fraudulent']
)

# 2. Split that 20% into half (10% Val, 10% Test)
# FIX: Use temp_data['fraudulent'] for stratification
val_data, test_data = train_test_split(
    temp_data, test_size=0.5, random_state=42, stratify=temp_data['fraudulent']
)

In [ ]:
X_train = train_data['full_text']
y_train = train_data['fraudulent']

X_val = val_data['full_text']
y_val = val_data['fraudulent']

X_test = test_data['full_text']
y_test = test_data['fraudulent']

### Tokenization

This sections serves to tokenize free text into a sequence of integers

In [ ]:
# Tokenize
def tokenize(text):
    return word_tokenize(text.lower())

# Build vocab from training data only
counter = Counter()
for text in X_train:
    counter.update(tokenize(text))

vocab = {"<unk>": 0, "<pad>": 1}
for word, freq in counter.items():
    if freq >= 1:   # raise to e.g. 2 to filter rare words
        vocab[word] = len(vocab)

# Convert text to integer sequences
def text_pipeline(text):
    return [vocab.get(token, vocab["<unk>"]) for token in tokenize(text)]

X_train_tok = [text_pipeline(text) for text in X_train]
X_val_tok   = [text_pipeline(text) for text in X_val]
X_test_tok  = [text_pipeline(text) for text in X_test]

print(f"Vocab size: {len(vocab)}")
print(f"Example: {X_train[0]}")
print(f"Tokenized: {X_train_tok[0]}")

### Embeddings

This section serves to convert the token ids into a high-dimensional vector to capture the semantic and syntactic meaning of the tokens

We will try a pretrained CBow as well as using FastText from scratch and evaluate their performance. 

#### FastText (using skipgram)

In [ ]:
sentences = [word_tokenize(text.lower()) for text in X_train]  
fasttext_model = FastText(vector_size=100, window=5, min_count=5, sg=1) #assuming we want a 100 word vector
fasttext_model.build_vocab(sentences)
fasttext_model.train(sentences, total_examples=fasttext_model.corpus_count, epochs=10)

Some checks if we we manage to learn technical jargon and correlated words

In [ ]:
print(fasttext_model.wv['saas']) #technical jargon
print(fasttext_model.wv.vectors.shape) 

In [ ]:
print(fasttext_model.wv['bingsu']) #checking for oov
print(fasttext_model.wv.vectors.shape) 

In [ ]:
print(fasttext_model.wv.similarity('software', 'engineer'))
print(fasttext_model.wv.similarity('skills', 'experience'))
fasttext_model.wv.most_similar('water', topn=10)


#### CBOW

In [ ]:

cbow_model = Word2Vec(vector_size=100, window=5, min_count=5, sg=0) #assuming we want a 100 word vector
cbow_model.build_vocab(sentences)
cbow_model.train(sentences, total_examples=cbow_model.corpus_count, epochs=10)

In [ ]:
print(cbow_model.wv.similarity('software', 'engineer'))
print(cbow_model.wv.similarity('skills', 'experience'))
cbow_model.wv.most_similar('research', topn=10)


### Hyperparameter tuning

We are going to tune the `sliding window size`, `embedding vector size`,`epochs`, `negative sampling` to obtain the optimal custom embedding which would be easily plugged into our model

we will measure through extrinsic evaluation and intrinsic evaluation

In [ ]:
#to check oov rate
def oov_rate(model, corpus):
    oov = sum(1 for w in corpus if w not in model.wv)
    return oov/len(corpus)

#check top 10 words are similar to each other
def nearest_neighbour(model, test_words, topn = 10):
    scores = []
    for word in test_words:
        try:
            neighbours = model.wv.most_similar(word, topn=topn)
            scores.append(np.mean([score for _, score in neighbours]))
        except KeyError:
            pass
    return np.mean(scores)

#check if 2 correlated and 2 uncorrelated words are similar
def analogy_score(model, test_cases):
    correct = 0
    for pos1, pos2, neg1, expected in test_cases:
        try:
            results = model.wv.most_similar(
                positive=[pos1, pos2], negative=[neg1], topn=5
            )
            predicted = [w for w, _ in results]
            if expected in predicted:
                correct += 1
        except KeyError:
            pass
    return correct / len(test_cases)

In [ ]:
#DO NOT RUN THIS IT WILL TAKE 10.5 HOURS

param_grid = {
    'vector_size': [100, 200, 300],
    'window':      [3, 5, 10],
    'min_count':   [2, 5],
    'epochs':      [10, 20],
    'negative':    [5, 10],
}

test_words = ['engineer', 'manager', 'python', 'healthcare', 'experience', 'salary']
job_analogies = [
    ('engineer', 'python', 'manager', 'java'),
    ('senior', 'engineer', 'junior', 'developer'),
    ('full_time', 'salary', 'part_time', 'hourly'),
    ('healthcare', 'nurse', 'finance', 'analyst'),
]

results = []

keys = list(param_grid.keys())
combos = list(product(*param_grid.values()))
print(f"Total combinations: {len(combos)} x 2 models = {len(combos)*2} runs")

for combo in combos:
    params = dict(zip(keys, combo))

    for model_type in ['fasttext', 'cbow']:
        if model_type == 'fasttext':
            model = FastText(**params, sg=1, min_n=3, max_n=6)
        else:
            model = Word2Vec(**params, sg=0)

        model.build_vocab(sentences)
        model.train(sentences, total_examples=model.corpus_count, epochs=params['epochs'])

        coherence = nearest_neighbour(model, test_words)
        oov       = oov_rate(model, vocab)
        analogy   = analogy_score(model, job_analogies)

        results.append({
            **params,
            'model_type': model_type,
            'coherence':  round(coherence, 4),
            'oov_rate':   round(oov, 4),
            'analogy':    round(analogy, 4),
        })
        print(f"[{model_type}] {params} → coherence={coherence:.4f}, oov={oov:.4f}, analogy={analogy:.4f}")

results_df = pd.DataFrame(results)
results_df.to_csv('data/clean/embedding_tuning.csv', index=False)